In [40]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42

TRAIN_PATH = "/content/drive/MyDrive/dataset_C_training.csv"
TEST_PATH = "/content/drive/MyDrive/dataset_C_testing.csv"

TARGET_COL = "covid_vaccine"
ID_COL = "respondent_id"

In [41]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

Training shape: (4756, 31)
Testing shape: (4749, 30)


,respondent_id,covid_concern,covid_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,employment_status,census_msa,household_adults,household_children,doctor_recc_covid,opinion_covid_vacc_effective,opinion_covid_risk,opinion_covid_sick_from_vacc,employment_sector,covid_vaccine
0,1,3.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,Employed,"MSA, Principle City",3.0,2.0,0,4,4,2.0,construction,0
1,2,2.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Employed,Non-MSA,0.0,0.0,0,5,2,1.0,education,1
2,3,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Employed,"MSA, Not Principle City",0.0,0.0,0,2,2,5.0,wholesale,0
3,4,2.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Not in Labor Force,"MSA, Not Principle City",1.0,0.0,1,3,3,2.0,NaN,1
4,5,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,Employed,"MSA, Not Principle City",0.0,0.0,0,3,2,2.0,wholesale,0


,respondent_id,covid_concern,covid_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,rent_or_own,employment_status,census_msa,household_adults,household_children,doctor_recc_covid,opinion_covid_vacc_effective,opinion_covid_risk,opinion_covid_sick_from_vacc,employment_sector
0,4757,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,Own,Not in Labor Force,"MSA, Principle City",0.0,0.0,0,4,2,2.0,NaN
1,4758,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,NaN,NaN,"MSA, Not Principle City",0.0,0.0,0,2,3,2.0,NaN
2,4759,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,Own,Not in Labor Force,"MSA, Principle City",1.0,0.0,0,3,2,2.0,NaN
3,4760,3.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,Rent,Employed,Non-MSA,1.0,0.0,0,4,4,1.0,agriculture
4,4761,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Own,Employed,Non-MSA,0.0,0.0,0,3,2,1.0,wholesale


In [42]:
train_df.columns

Index(['respondent_id', 'covid_concern', 'covid_knowledge',
       'behavioral_antiviral_meds', 'behavioral_avoidance',
       'behavioral_face_mask', 'behavioral_wash_hands',
       'behavioral_large_gatherings', 'behavioral_outside_home',
       'behavioral_touch_face', 'chronic_med_condition',
       'child_under_6_months', 'health_worker', 'health_insurance',
       'age_group', 'education', 'race', 'sex', 'income_poverty',
       'marital_status', 'rent_or_own', 'employment_status', 'census_msa',
       'household_adults', 'household_children', 'doctor_recc_covid',
       'opinion_covid_vacc_effective', 'opinion_covid_risk',
       'opinion_covid_sick_from_vacc', 'employment_sector', 'covid_vaccine'],
      dtype='object')

In [43]:
print("Target distribution:")
display(train_df[TARGET_COL].value_counts())
display(train_df[TARGET_COL].value_counts(normalize=True).rename("proportion"))

print("\nMissing values in training set:")
missing_train = (train_df.isna().mean().sort_values(ascending=False).rename("missing_rate").to_frame())
display(missing_train[missing_train["missing_rate"] > 0])

print("\nMissing values in testing set:")
missing_test = (test_df.isna().mean().sort_values(ascending=False).rename("missing_rate").to_frame())
display(missing_test[missing_test["missing_rate"] > 0])

print("\nData types:")
display(train_df.dtypes)

Target distribution:


,count
covid_vaccine,
0,3202
1,1554


,proportion
covid_vaccine,
0,0.673255
1,0.326745



Missing values in training set:


,missing_rate
employment_sector,0.489907
health_insurance,0.401177
income_poverty,0.157275
rent_or_own,0.069176
employment_status,0.050042
marital_status,0.048570
education,0.047939
chronic_med_condition,0.035534
child_under_6_months,0.028385
health_worker,0.027754



Missing values in testing set:


,missing_rate
employment_sector,0.506422
health_insurance,0.417351
income_poverty,0.167825
rent_or_own,0.079806
employment_status,0.058328
marital_status,0.056644
education,0.056644
chronic_med_condition,0.035376
child_under_6_months,0.032638
health_worker,0.031375



Data types:


,0
respondent_id,int64
covid_concern,float64
covid_knowledge,float64
behavioral_antiviral_meds,float64
behavioral_avoidance,float64
behavioral_face_mask,float64
behavioral_wash_hands,float64
behavioral_large_gatherings,float64
behavioral_outside_home,float64
behavioral_touch_face,float64


In [44]:
def clean_hidden_missing_values(df):
    df = df.copy()

    hidden_missing_values = [
        "", " ", "NA", "N/A", "na", "n/a",
        "NULL", "null", "None", "none",
        "?", "-", "--"
    ]

    df = df.replace(hidden_missing_values, np.nan)
    return df


train_df = clean_hidden_missing_values(train_df)
test_df = clean_hidden_missing_values(test_df)
print("Hidden missing values cleaned.")

Hidden missing values cleaned.


In [45]:
X_full = train_df.drop(columns=[TARGET_COL])
y_full = train_df[TARGET_COL].astype(int)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(X_full, y_full, test_size=0.20, stratify=y_full, random_state=RANDOM_STATE)
X_test_raw = test_df.copy()

print("X_train_raw:", X_train_raw.shape)
print("X_val_raw:", X_val_raw.shape)
print("X_test_raw:", X_test_raw.shape)

print("\nTraining target balance:")
display(y_train.value_counts(normalize=True))
print("\nValidation target balance:")
display(y_val.value_counts(normalize=True))

X_train_raw: (3804, 30)
X_val_raw: (952, 30)
X_test_raw: (4749, 30)

Training target balance:


,proportion
covid_vaccine,
0,0.673239
1,0.326761



Validation target balance:


,proportion
covid_vaccine,
0,0.673319
1,0.326681


In [46]:
train_ids = X_train_raw[ID_COL].copy()
val_ids = X_val_raw[ID_COL].copy()
test_ids = X_test_raw[ID_COL].copy()
X_train_raw = X_train_raw.drop(columns=[ID_COL])
X_val_raw = X_val_raw.drop(columns=[ID_COL])
X_test_raw = X_test_raw.drop(columns=[ID_COL])

print("ID removed from features.")
print("X_train_raw:", X_train_raw.shape)
print("X_val_raw:", X_val_raw.shape)
print("X_test_raw:", X_test_raw.shape)

ID removed from features.
X_train_raw: (3804, 29)
X_val_raw: (952, 29)
X_test_raw: (4749, 29)


In [47]:
behavior_cols = [
    "behavioral_antiviral_meds",
    "behavioral_avoidance",
    "behavioral_face_mask",
    "behavioral_wash_hands",
    "behavioral_large_gatherings",
    "behavioral_outside_home",
    "behavioral_touch_face"
]

opinion_cols = [
    "opinion_covid_vacc_effective",
    "opinion_covid_risk",
    "opinion_covid_sick_from_vacc"
]

high_missing_cols = [
    "employment_sector",
    "health_insurance",
    "income_poverty"
]


def add_missing_aware_features(df):
    df = df.copy()
    for col in high_missing_cols:
        if col in df.columns:
            df[f"{col}_was_missing"] = df[col].isna().astype(int)
    existing_behavior_cols = [col for col in behavior_cols if col in df.columns]
    df["behavior_score"] = df[existing_behavior_cols].sum(axis=1, skipna=True)
    df["behavior_missing_count"] = df[existing_behavior_cols].isna().sum(axis=1)
    df["behavior_observed_count"] = df[existing_behavior_cols].notna().sum(axis=1)

    existing_opinion_cols = [col for col in opinion_cols if col in df.columns]
    df["opinion_missing_count"] = df[existing_opinion_cols].isna().sum(axis=1)
    df["opinion_observed_count"] = df[existing_opinion_cols].notna().sum(axis=1)
    df["covid_opinion_positive_score"] = (df["opinion_covid_vacc_effective"].fillna(0) + df["opinion_covid_risk"].fillna(0) - df["opinion_covid_sick_from_vacc"].fillna(0))

    household_cols = ["household_adults", "household_children"]
    existing_household_cols = [col for col in household_cols if col in df.columns]
    df["household_total"] = df[existing_household_cols].sum(axis=1, skipna=True)
    df["household_missing_count"] = df[existing_household_cols].isna().sum(axis=1)
    df["doctor_recc_x_risk"] = (df["doctor_recc_covid"].fillna(0) * df["opinion_covid_risk"].fillna(0))

    df["behavior_score_bin"] = pd.cut(df["behavior_score"], bins=[-0.1, 1, 3, 7], labels=["low_behavior", "medium_behavior", "high_behavior"]).astype("object")
    df["household_total_bin"] = pd.cut(df["household_total"], bins=[-0.1, 0, 2, 10], labels=["alone", "small_household", "large_household"]).astype("object")

    return df


X_train_fe = add_missing_aware_features(X_train_raw)
X_val_fe = add_missing_aware_features(X_val_raw)
X_test_fe = add_missing_aware_features(X_test_raw)

print("Feature engineering complete.")
print("X_train_fe:", X_train_fe.shape)
print("X_val_fe:", X_val_fe.shape)
print("X_test_fe:", X_test_fe.shape)

Feature engineering complete.
X_train_fe: (3804, 43)
X_val_fe: (952, 43)
X_test_fe: (4749, 43)


In [48]:
text_categorical_cols = [
    "age_group",
    "education",
    "race",
    "sex",
    "income_poverty",
    "marital_status",
    "rent_or_own",
    "employment_status",
    "census_msa",
    "employment_sector"
]

survey_numeric_cols = [
    "covid_concern",
    "covid_knowledge",
    "behavioral_antiviral_meds",
    "behavioral_avoidance",
    "behavioral_face_mask",
    "behavioral_wash_hands",
    "behavioral_large_gatherings",
    "behavioral_outside_home",
    "behavioral_touch_face",
    "chronic_med_condition",
    "child_under_6_months",
    "health_worker",
    "health_insurance",
    "doctor_recc_covid",
    "opinion_covid_vacc_effective",
    "opinion_covid_risk",
    "opinion_covid_sick_from_vacc",
    "household_adults",
    "household_children"
]

binned_categorical_cols = [
    "behavior_score_bin",
    "household_total_bin"
]

numeric_passthrough_cols = [
    "employment_sector_was_missing",
    "health_insurance_was_missing",
    "income_poverty_was_missing",
    "behavior_score",
    "behavior_missing_count",
    "behavior_observed_count",
    "opinion_missing_count",
    "opinion_observed_count",
    "covid_opinion_positive_score",
    "household_total",
    "household_missing_count",
    "doctor_recc_x_risk"
]

encoding_cols = [
    col for col in
    text_categorical_cols + survey_numeric_cols + binned_categorical_cols
    if col in X_train_fe.columns
]
numeric_passthrough_cols = [
    col for col in numeric_passthrough_cols
    if col in X_train_fe.columns
]

print("Columns to one-hot encode:", len(encoding_cols))
print(encoding_cols)
print("\nNumeric passthrough columns:", len(numeric_passthrough_cols))
print(numeric_passthrough_cols)

Columns to one-hot encode: 31
['age_group', 'education', 'race', 'sex', 'income_poverty', 'marital_status', 'rent_or_own', 'employment_status', 'census_msa', 'employment_sector', 'covid_concern', 'covid_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home', 'behavioral_touch_face', 'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance', 'doctor_recc_covid', 'opinion_covid_vacc_effective', 'opinion_covid_risk', 'opinion_covid_sick_from_vacc', 'household_adults', 'household_children', 'behavior_score_bin', 'household_total_bin']

Numeric passthrough columns: 12
['employment_sector_was_missing', 'health_insurance_was_missing', 'income_poverty_was_missing', 'behavior_score', 'behavior_missing_count', 'behavior_observed_count', 'opinion_missing_count', 'opinion_observed_count', 'covid_opinion_positive_score', 'household_total', 'household_missing

In [49]:
def make_string_missing_transformer(columns):
    def transform(X):
        X_df = pd.DataFrame(X, columns=columns).copy()
        X_df = X_df.astype("object")
        X_df = X_df.where(~X_df.isna(), "Missing")
        X_df = X_df.astype(str)
        return X_df

    return FunctionTransformer(transform, validate=False)


def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )


missing_as_category_pipeline = Pipeline(steps=[("to_string_missing", make_string_missing_transformer(encoding_cols)),("onehot", make_onehot_encoder())])
print("Missing-as-category encoding pipeline created.")

Missing-as-category encoding pipeline created.


In [50]:
preprocessor_onehot_missing = ColumnTransformer(transformers=[("encoded", missing_as_category_pipeline, encoding_cols), ("numeric", "passthrough", numeric_passthrough_cols)], remainder="drop")
print("Full preprocessor created.")

Full preprocessor created.


In [51]:
X_train_encoded = preprocessor_onehot_missing.fit_transform(X_train_fe)
X_val_encoded = preprocessor_onehot_missing.transform(X_val_fe)
X_test_encoded = preprocessor_onehot_missing.transform(X_test_fe)

print("X_train_encoded shape:", X_train_encoded.shape)
print("X_val_encoded shape:", X_val_encoded.shape)
print("X_test_encoded shape:", X_test_encoded.shape)

X_train_encoded shape: (3804, 145)
X_val_encoded shape: (952, 145)
X_test_encoded shape: (4749, 145)


In [52]:
onehot = preprocessor_onehot_missing.named_transformers_["encoded"].named_steps["onehot"]
onehot_feature_names = onehot.get_feature_names_out(encoding_cols).tolist()
final_feature_names = onehot_feature_names + numeric_passthrough_cols
print("Number of final features:", len(final_feature_names))
print(final_feature_names[:30])

Number of final features: 145
['age_group_18 - 34 Years', 'age_group_35 - 44 Years', 'age_group_45 - 54 Years', 'age_group_55 - 64 Years', 'age_group_65+ Years', 'education_12 Years', 'education_< 12 Years', 'education_College Graduate', 'education_Missing', 'education_Some College', 'race_Black', 'race_Hispanic', 'race_Other or Multiple', 'race_White', 'sex_Female', 'sex_Male', 'income_poverty_<= $75,000, Above Poverty', 'income_poverty_> $75,000', 'income_poverty_Below Poverty', 'income_poverty_Missing', 'marital_status_Married', 'marital_status_Missing', 'marital_status_Not Married', 'rent_or_own_Missing', 'rent_or_own_Own', 'rent_or_own_Rent', 'employment_status_Employed', 'employment_status_Missing', 'employment_status_Not in Labor Force', 'employment_status_Unemployed']


In [53]:
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=final_feature_names, index=X_train_fe.index)
X_val_encoded_df = pd.DataFrame(X_val_encoded, columns=final_feature_names, index=X_val_fe.index)
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=final_feature_names, index=X_test_fe.index)
print("Encoded training frame:")
display(X_train_encoded_df.head())
print("Encoded validation frame:")
display(X_val_encoded_df.head())

Encoded training frame:


,age_group_18 - 34 Years,age_group_35 - 44 Years,age_group_45 - 54 Years,age_group_55 - 64 Years,age_group_65+ Years,education_12 Years,education_< 12 Years,education_College Graduate,education_Missing,education_Some College,...,income_poverty_was_missing,behavior_score,behavior_missing_count,behavior_observed_count,opinion_missing_count,opinion_observed_count,covid_opinion_positive_score,household_total,household_missing_count,doctor_recc_x_risk
909,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,3.0,0.0,7.0,0.0,3.0,5.0,4.0,0.0,0.0
3586,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,2.0,0.0,7.0,0.0,3.0,6.0,2.0,0.0,3.0
3700,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,5.0,0.0,7.0,0.0,3.0,1.0,1.0,0.0,2.0
1927,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,7.0,0.0,3.0,4.0,0.0,0.0,0.0
4552,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,5.0,0.0,7.0,0.0,3.0,5.0,3.0,0.0,5.0


Encoded validation frame:


,age_group_18 - 34 Years,age_group_35 - 44 Years,age_group_45 - 54 Years,age_group_55 - 64 Years,age_group_65+ Years,education_12 Years,education_< 12 Years,education_College Graduate,education_Missing,education_Some College,...,income_poverty_was_missing,behavior_score,behavior_missing_count,behavior_observed_count,opinion_missing_count,opinion_observed_count,covid_opinion_positive_score,household_total,household_missing_count,doctor_recc_x_risk
145,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,3.0,0.0,7.0,0.0,3.0,2.0,4.0,0.0,0.0
72,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,4.0,0.0,7.0,0.0,3.0,3.0,1.0,0.0,0.0
2156,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,7.0,0.0,3.0,5.0,2.0,0.0,0.0
3542,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,5.0,0.0,7.0,0.0,3.0,4.0,1.0,0.0,4.0
2889,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,6.0,0.0,7.0,0.0,3.0,7.0,0.0,2.0,0.0


In [54]:
print("Original training rows after split:", X_train_fe.shape[0])
print("Encoded training rows:", X_train_encoded_df.shape[0])

print("\nOriginal validation rows:", X_val_fe.shape[0])
print("Encoded validation rows:", X_val_encoded_df.shape[0])

print("\nOriginal test rows:", X_test_fe.shape[0])
print("Encoded test rows:", X_test_encoded_df.shape[0])

print("\nMissing values after encoding:")
print("Train:", X_train_encoded_df.isna().sum().sum())
print("Validation:", X_val_encoded_df.isna().sum().sum())
print("Test:", X_test_encoded_df.isna().sum().sum())

missing_encoded_cols = [col for col in X_train_encoded_df.columns if "Missing" in col]
print("\nNumber of encoded Missing columns:", len(missing_encoded_cols))
print(missing_encoded_cols[:30])

Original training rows after split: 3804
Encoded training rows: 3804

Original validation rows: 952
Encoded validation rows: 952

Original test rows: 4749
Encoded test rows: 4749

Missing values after encoding:
Train: 0
Validation: 0
Test: 0

Number of encoded Missing columns: 22
['education_Missing', 'income_poverty_Missing', 'marital_status_Missing', 'rent_or_own_Missing', 'employment_status_Missing', 'employment_sector_Missing', 'covid_concern_Missing', 'covid_knowledge_Missing', 'behavioral_antiviral_meds_Missing', 'behavioral_avoidance_Missing', 'behavioral_face_mask_Missing', 'behavioral_wash_hands_Missing', 'behavioral_large_gatherings_Missing', 'behavioral_outside_home_Missing', 'behavioral_touch_face_Missing', 'chronic_med_condition_Missing', 'child_under_6_months_Missing', 'health_worker_Missing', 'health_insurance_Missing', 'opinion_covid_sick_from_vacc_Missing', 'household_adults_Missing', 'household_children_Missing']


In [55]:
age_categories = [
    "Missing",
    "18 - 34 Years",
    "35 - 44 Years",
    "45 - 54 Years",
    "55 - 64 Years",
    "65+ Years"
]

education_categories = [
    "Missing",
    "< 12 Years",
    "12 Years",
    "Some College",
    "College Graduate"
]

income_categories = [
    "Missing",
    "Below Poverty",
    "<= $75,000, Above Poverty",
    "> $75,000"
]


def add_manual_ordinal_features(df):
    df = df.copy()

    age_map = {v: i for i, v in enumerate(age_categories)}
    education_map = {v: i for i, v in enumerate(education_categories)}
    income_map = {v: i for i, v in enumerate(income_categories)}

    if "age_group" in df.columns:
        df["age_group_ordinal_encoded"] = (
            df["age_group"]
            .fillna("Missing")
            .map(age_map)
            .fillna(-1)
            .astype(int)
        )

    if "education" in df.columns:
        df["education_ordinal_encoded"] = (
            df["education"]
            .fillna("Missing")
            .map(education_map)
            .fillna(-1)
            .astype(int)
        )

    if "income_poverty" in df.columns:
        df["income_poverty_ordinal_encoded"] = (
            df["income_poverty"]
            .fillna("Missing")
            .map(income_map)
            .fillna(-1)
            .astype(int)
        )

    return df


X_train_hybrid = add_manual_ordinal_features(X_train_fe)
X_val_hybrid = add_manual_ordinal_features(X_val_fe)
X_test_hybrid = add_manual_ordinal_features(X_test_fe)

print("Manual ordinal features added.")

Manual ordinal features added.


In [56]:
ordinal_created_cols = [
    "age_group_ordinal_encoded",
    "education_ordinal_encoded",
    "income_poverty_ordinal_encoded"
]

ordinal_created_cols = [
    col for col in ordinal_created_cols
    if col in X_train_hybrid.columns
]

ordered_text_cols = ["age_group", "education", "income_poverty"]
hybrid_onehot_cols = [
    col for col in encoding_cols
    if col not in ordered_text_cols
]

hybrid_numeric_cols = numeric_passthrough_cols + ordinal_created_cols
hybrid_onehot_pipeline = Pipeline(steps=[
    ("to_string_missing", make_string_missing_transformer(hybrid_onehot_cols)),
    ("onehot", make_onehot_encoder())
])
preprocessor_hybrid_ordinal_onehot = ColumnTransformer(
    transformers=[
        ("onehot", hybrid_onehot_pipeline, hybrid_onehot_cols),
        ("numeric", "passthrough", hybrid_numeric_cols)
    ],
    remainder="drop"
)

print("Hybrid ordinal + one-hot preprocessor created.")
print("One-hot columns:", len(hybrid_onehot_cols))
print("Numeric/ordinal passthrough columns:", len(hybrid_numeric_cols))

Hybrid ordinal + one-hot preprocessor created.
One-hot columns: 28
Numeric/ordinal passthrough columns: 15


In [57]:
X_train_hybrid_encoded = preprocessor_hybrid_ordinal_onehot.fit_transform(X_train_hybrid)
X_val_hybrid_encoded = preprocessor_hybrid_ordinal_onehot.transform(X_val_hybrid)
X_test_hybrid_encoded = preprocessor_hybrid_ordinal_onehot.transform(X_test_hybrid)

print("X_train_hybrid_encoded:", X_train_hybrid_encoded.shape)
print("X_val_hybrid_encoded:", X_val_hybrid_encoded.shape)
print("X_test_hybrid_encoded:", X_test_hybrid_encoded.shape)

X_train_hybrid_encoded: (3804, 134)
X_val_hybrid_encoded: (952, 134)
X_test_hybrid_encoded: (4749, 134)


In [58]:
X_train_model = X_train_encoded_df.copy()
X_val_model = X_val_encoded_df.copy()
X_test_model = X_test_encoded_df.copy()

print("Ready for modelling.")
print("X_train_model:", X_train_model.shape)
print("X_val_model:", X_val_model.shape)
print("X_test_model:", X_test_model.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

Ready for modelling.
X_train_model: (3804, 145)
X_val_model: (952, 145)
X_test_model: (4749, 145)
y_train: (3804,)
y_val: (952,)


In [63]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier
)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42

In [64]:
def get_prediction_scores(model, X):
    """
    Returns positive-class probability scores.
    Uses predict_proba if available, otherwise decision_function.
    """
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        return scores

    return model.predict(X)


def evaluate_and_save_model(
    model_name,
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    test_ids,
    filename_prefix
):
    """
    Fits one model, evaluates on validation data, saves test predictions.
    """
    print("=" * 80)
    print("Training:", model_name)

    model.fit(X_train, y_train)

    val_scores = get_prediction_scores(model, X_val)
    val_preds = (val_scores >= 0.5).astype(int)

    metrics = {
        "model": model_name,
        "roc_auc": roc_auc_score(y_val, val_scores),
        "accuracy": accuracy_score(y_val, val_preds),
        "precision": precision_score(y_val, val_preds, zero_division=0),
        "recall": recall_score(y_val, val_preds, zero_division=0),
        "f1": f1_score(y_val, val_preds, zero_division=0)
    }

    print("ROC-AUC:", round(metrics["roc_auc"], 4))
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("Precision:", round(metrics["precision"], 4))
    print("Recall:", round(metrics["recall"], 4))
    print("F1:", round(metrics["f1"], 4))

    print("\nClassification report:")
    print(classification_report(y_val, val_preds))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_val, val_preds))

    test_scores = get_prediction_scores(model, X_test)

    submission = pd.DataFrame({
        "respondent_id": test_ids,
        "covid_vaccine": test_scores
    })

    output_file = f"{filename_prefix}_{model_name}.csv"
    submission.to_csv(output_file, index=False)

    print("\nSaved test prediction file:", output_file)

    return metrics, model, val_scores, submission

In [62]:
def get_prediction_scores(model, X):
    """
    Returns probability-like scores for the positive class.
    Uses predict_proba if available, otherwise decision_function.
    """
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        return scores

    return model.predict(X)


def evaluate_model(model_name, model, X_train, y_train, X_val, y_val, threshold=0.5):
    """
    Fits model and evaluates it on validation data.
    """
    model.fit(X_train, y_train)

    val_scores = get_prediction_scores(model, X_val)
    val_preds = (val_scores >= threshold).astype(int)

    metrics = {
        "model": model_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_val, val_scores),
        "accuracy": accuracy_score(y_val, val_preds),
        "precision": precision_score(y_val, val_preds, zero_division=0),
        "recall": recall_score(y_val, val_preds, zero_division=0),
        "f1": f1_score(y_val, val_preds, zero_division=0)
    }

    return metrics, val_scores, val_preds


def find_best_threshold(y_true, scores):
    """
    Finds threshold that maximises F1 score on validation data.
    """
    thresholds = np.arange(0.05, 0.96, 0.01)

    best_threshold = 0.5
    best_f1 = -1

    for threshold in thresholds:
        preds = (scores >= threshold).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

In [65]:
thivas_models = {
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.03,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=RANDOM_STATE
    ),

    "mlp_neural_network": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=500,
            early_stopping=True,
            random_state=RANDOM_STATE
        ))
    ])
}

thivas_results = []

for model_name, model in thivas_models.items():
    metrics, fitted_model, val_scores, submission = evaluate_and_save_model(
        model_name=model_name,
        model=model,
        X_train=X_train_model,
        y_train=y_train,
        X_val=X_val_model,
        y_val=y_val,
        X_test=X_test_model,
        test_ids=test_ids,
        filename_prefix="person06"
    )

    thivas_results.append(metrics)

person06_results_df = pd.DataFrame(thivas_results)
display(person06_results_df)

person06_results_df.to_csv("person06_results.csv", index=False)

Training: hist_gradient_boosting
ROC-AUC: 0.8472
Accuracy: 0.8099
Precision: 0.7425
Recall: 0.6399
F1: 0.6874

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86       641
           1       0.74      0.64      0.69       311

    accuracy                           0.81       952
   macro avg       0.79      0.77      0.78       952
weighted avg       0.81      0.81      0.81       952


Confusion matrix:
[[572  69]
 [112 199]]

Saved test prediction file: person06_hist_gradient_boosting.csv
Training: mlp_neural_network
ROC-AUC: 0.8449
Accuracy: 0.8036
Precision: 0.7403
Recall: 0.6141
F1: 0.6714

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.90      0.86       641
           1       0.74      0.61      0.67       311

    accuracy                           0.80       952
   macro avg       0.78      0.75      0.77       952
weighted avg       0.80      0

,model,roc_auc,accuracy,precision,recall,f1
0,hist_gradient_boosting,0.847214,0.809874,0.742537,0.639871,0.687392
1,mlp_neural_network,0.844897,0.803571,0.740310,0.614148,0.671353
